# Stage 1 — Final Anomaly Detector (clean rebuild)

Combines two independent detection paths:
1. **Texture rule** (original 5-feature GLCM/LBP, 2-of-5 agreement, bucket-relative z-scores)
2. **Local brightness rule** (NEW — flags a region if it's a brightness outlier relative to
   the REST OF THE SAME IMAGE, no external baseline needed). This specifically catches bright
   foam and dark shadows that the texture-only rule structurally cannot see.

A detection fires if EITHER path finds something (OR logic).

**Final locked parameters** (chosen after cross-dataset validation across prawn/tuna/cetacean):
- Texture rule: `z_thresh=2.5, min_features_flagged=2, min_component_area_px=2000`
- Local-brightness rule: `local_z_thresh=2.5, min_area_local=1000` (lowered from 2000 to raise tuna further)
- **Expected result: ~46.22% false-positive rate, 100% prawn, ~94% tuna, 100% cetacean**

This is a genuine trade-off, not free: false-positive rate is higher than the original
prawn-only-tuned config (11.90%), in exchange for tuna detection going from 5.5% to ~87%,
with prawn and cetacean unaffected. Stage 2 is expected to correct a large share of the
extra false positives (measured at ~91% correction in earlier connected-pipeline testing),
though that correction rate has not yet been re-validated at this specific operating point.

In [8]:
# --- Cell 1: Imports and config ---
import json
import pickle
import random
from pathlib import Path

import cv2
import numpy as np
from skimage.segmentation import slic
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from scipy import ndimage

# --- Dataset paths: update to match your machine ---
BASELINE_DIR = Path(r"C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_01\data\normal_sea\baseline_split")
NEGATIVE_DIR = Path(r"C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_01\data\normal_sea\heldout_split")
PRAWN_DIR    = Path(r"C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_01\data\prawns")
TUNA_DIR     = Path(r"C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_01\data\tuna")          # ~2,000 raw tuna images
CETACEAN_DIR = Path(r"C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_01\data\cetacean")   # ~7,460 annotated cetacean images

CROSS_VALIDATION_SAMPLE_SIZE = 200   # tuna/cetacean sample size for the final confirmation run

# --- Pre-processing config ---
IMG_SIZE = (512, 512)
GLINT_THRESH = 220
GLCM_LEVELS = 32

# --- SLIC config ---
N_SEGMENTS = 250
COMPACTNESS = 20

# --- Sea-state bucketing ---
N_BUCKETS = 3

# --- FINAL detector parameters (locked) ---
Z_THRESH = 2.5                  # texture rule
MIN_FEATURES_FLAGGED = 2        # texture rule: how many of 5 features must agree
MIN_COMPONENT_AREA_PX = 2000    # texture rule: min contiguous flagged area
LOCAL_Z_THRESH = 2.5            # local-brightness rule
MIN_AREA_LOCAL = 1000           # local-brightness rule: min contiguous flagged area (lowered from 2000)

FEATURE_KEYS = ["contrast_L", "homogeneity_L", "energy_L", "contrast_a", "lbp_var"]

OUTPUT_DIR = Path("stage1_output")
OUTPUT_DIR.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

In [2]:
# --- Cell 2: Pre-processing functions ---

def load_and_resize(path, size=IMG_SIZE):
    img = cv2.imread(str(path))
    img = cv2.resize(img, size)
    return img

def suppress_glint(img_bgr, thresh=GLINT_THRESH, dilate_iter=2):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    L = lab[:, :, 0]
    glint_mask = (L > thresh).astype(np.uint8)
    kernel = np.ones((5, 5), np.uint8)
    glint_mask = cv2.dilate(glint_mask, kernel, iterations=dilate_iter)
    glint_mask = cv2.morphologyEx(glint_mask, cv2.MORPH_CLOSE, kernel)
    return glint_mask

def preprocess_image(path, glint_thresh=GLINT_THRESH):
    img = load_and_resize(path)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    glint_mask = suppress_glint(img, thresh=glint_thresh)
    return img, lab, glint_mask

In [3]:
# --- Cell 3: Feature extraction (SLIC + GLCM + LBP), includes mean_L per
#     superpixel for the local-brightness rule ---

def quantize(patch, levels=GLCM_LEVELS):
    return (patch.astype(np.float32) / 256 * levels).astype(np.uint8)

def glcm_props(patch_q, levels=GLCM_LEVELS):
    glcm = graycomatrix(patch_q, distances=[1], angles=[0], levels=levels, symmetric=True, normed=True)
    return (float(graycoprops(glcm, 'contrast')[0, 0]),
            float(graycoprops(glcm, 'homogeneity')[0, 0]),
            float(graycoprops(glcm, 'energy')[0, 0]))

def extract_superpixel_features(img_lab, glint_mask, n_segments=N_SEGMENTS, compactness=COMPACTNESS):
    L_channel = img_lab[:, :, 0]
    a_channel = img_lab[:, :, 1]
    segments = slic(img_lab, n_segments=n_segments, compactness=compactness, start_label=1)
    objects = ndimage.find_objects(segments)

    features = []
    for seg_id, bbox in enumerate(objects, start=1):
        if bbox is None:
            continue
        y_slice, x_slice = bbox
        local_mask = segments[y_slice, x_slice] == seg_id
        if glint_mask[y_slice, x_slice][local_mask].mean() > 0.5:
            continue
        patch_L = L_channel[y_slice, x_slice]
        patch_a = a_channel[y_slice, x_slice]
        if patch_L.shape[0] < 2 or patch_L.shape[1] < 2:
            continue
        contrast_L, homogeneity_L, energy_L = glcm_props(quantize(patch_L))
        contrast_a, _, _ = glcm_props(quantize(patch_a))
        lbp = local_binary_pattern(patch_L, P=8, R=1, method='uniform')
        lbp_var = float(np.var(lbp))
        features.append({
            "seg_id": int(seg_id),
            "contrast_L": contrast_L, "homogeneity_L": homogeneity_L, "energy_L": energy_L,
            "contrast_a": contrast_a, "lbp_var": lbp_var,
            "mean_L": float(patch_L.mean()),   # used by the local-brightness rule only
        })
    return segments, features

def image_sea_state_score(img_lab, glint_mask):
    L = img_lab[:, :, 0]
    valid = glint_mask == 0
    vals = L[valid] if valid.any() else L.flatten()
    return float(np.mean(vals))

In [4]:
# --- Cell 4: Build the sea-state bucketed baseline ---
# Dataset used: ALL images in BASELINE_DIR.

all_baseline_paths = list(BASELINE_DIR.glob("*.jpg")) or list(BASELINE_DIR.glob("*.png"))
print(f"Building baseline from {len(all_baseline_paths)} normal sea images...")

baseline_scores = []
baseline_feats_per_image = []
for path in all_baseline_paths:
    img, lab, glint_mask = preprocess_image(path)
    segments, feats = extract_superpixel_features(lab, glint_mask)
    baseline_scores.append(image_sea_state_score(lab, glint_mask))
    baseline_feats_per_image.append(feats)

baseline_scores = np.array(baseline_scores)

def compute_bucket_edges(scores, n_buckets=N_BUCKETS):
    percentiles = np.linspace(0, 100, n_buckets + 1)[1:-1]
    return np.percentile(scores, percentiles)

def assign_bucket_from_score(score, edges):
    return int(np.searchsorted(edges, score))

bucket_edges = compute_bucket_edges(baseline_scores, N_BUCKETS)
bucket_labels = np.array([assign_bucket_from_score(s, bucket_edges) for s in baseline_scores])

bucket_ref = {}
for b in range(N_BUCKETS):
    stats = {k: [] for k in FEATURE_KEYS}
    for feats, label in zip(baseline_feats_per_image, bucket_labels):
        if label != b:
            continue
        for f in feats:
            for k in FEATURE_KEYS:
                stats[k].append(f[k])
    n_images_in_bucket = int(np.sum(bucket_labels == b))
    bucket_ref[b] = {k: (float(np.mean(v)), float(np.std(v))) for k, v in stats.items()}
    print(f"Bucket {b}: {n_images_in_bucket} images")

def assign_bucket(img_lab, glint_mask, edges=bucket_edges):
    score = image_sea_state_score(img_lab, glint_mask)
    return assign_bucket_from_score(score, edges)

print(f"Bucket edges: {bucket_edges}")

Building baseline from 3283 normal sea images...
Bucket 0: 1095 images
Bucket 1: 1094 images
Bucket 2: 1094 images
Bucket edges: [ 98.48828532 132.11243057]


In [11]:
# --- Cell 5: The combined detector - texture rule OR local-brightness rule ---

def score_region_deviation(feat, baseline_ref, z_thresh=Z_THRESH, min_features_flagged=MIN_FEATURES_FLAGGED):
    z_scores = {k: abs(feat[k] - baseline_ref[k][0]) / (baseline_ref[k][1] + 1e-6) for k in FEATURE_KEYS}
    n_exceeding = sum(1 for z in z_scores.values() if z > z_thresh)
    return n_exceeding >= min_features_flagged

def texture_rule_detected(segments, feats, baseline_ref, z_thresh=Z_THRESH,
                            min_area=MIN_COMPONENT_AREA_PX, min_features_flagged=MIN_FEATURES_FLAGGED):
    flagged_ids = {f["seg_id"] for f in feats if score_region_deviation(f, baseline_ref, z_thresh, min_features_flagged)}
    if not flagged_ids:
        return False
    flagged_mask = np.isin(segments, list(flagged_ids))
    labeled, n_components = ndimage.label(flagged_mask)
    if n_components == 0:
        return False
    sizes = ndimage.sum(flagged_mask, labeled, range(1, n_components + 1))
    return bool(sizes.max() >= min_area)

def local_brightness_rule_detected(feats, segments, local_z_thresh=LOCAL_Z_THRESH, min_area_local=MIN_AREA_LOCAL):
    """Flags a region if it's a brightness outlier relative to the REST OF THIS SAME IMAGE -
    no external baseline involved. Catches bright foam / dark shadows the texture rule misses."""
    if not feats:
        return False
    mean_Ls = np.array([f["mean_L"] for f in feats])
    img_mean, img_std = mean_Ls.mean(), mean_Ls.std()
    flagged_ids = {f["seg_id"] for f in feats if abs(f["mean_L"] - img_mean) / (img_std + 1e-6) > local_z_thresh}
    if not flagged_ids:
        return False
    flagged_mask = np.isin(segments, list(flagged_ids))
    labeled, n_components = ndimage.label(flagged_mask)
    if n_components == 0:
        return False
    sizes = ndimage.sum(flagged_mask, labeled, range(1, n_components + 1))
    return bool(sizes.max() >= min_area_local)

def run_detector(path, bucket_ref=bucket_ref, edges=bucket_edges):
    """The final combined detector for one image. Returns True if EITHER rule fires."""
    img, lab, glint_mask = preprocess_image(path)
    segments, feats = extract_superpixel_features(lab, glint_mask)
    bucket = assign_bucket(lab, glint_mask, edges)
    ref = bucket_ref[bucket]

    if texture_rule_detected(segments, feats, ref):
        return True
    return local_brightness_rule_detected(feats, segments)

def run_detector_on_dir(img_dir, sample_size=None):
    paths = list(Path(img_dir).glob("*.jpg")) or list(Path(img_dir).glob("*.png"))
    if sample_size is not None and sample_size < len(paths):
        paths = random.sample(paths, sample_size)
    if len(paths) == 0:
        raise ValueError(f"No images found in {img_dir}")
    detections = [run_detector(p) for p in paths]
    return float(np.mean(detections)), len(paths)

In [12]:
# --- Cell 6: Final validation across all four datasets ---
# Confirms the combined detector reproduces the expected result end-to-end,
# not just from cached sweep data.

false_positive_rate, n_negative = run_detector_on_dir(NEGATIVE_DIR)
prawn_rate, n_prawn = run_detector_on_dir(PRAWN_DIR)
tuna_rate, n_tuna = run_detector_on_dir(TUNA_DIR, sample_size=CROSS_VALIDATION_SAMPLE_SIZE)
cetacean_rate, n_cetacean = run_detector_on_dir(CETACEAN_DIR, sample_size=CROSS_VALIDATION_SAMPLE_SIZE)

print(f"{'Dataset':<20} | {'n':>5} | {'Rate':>8}")
print(f"{'Negative controls':<20} | {n_negative:>5} | {false_positive_rate:>7.2%}  (expect ~46.22%)")
print(f"{'Prawn':<20} | {n_prawn:>5} | {prawn_rate:>7.2%}  (expect 100.00%)")
print(f"{'Tuna':<20} | {n_tuna:>5} | {tuna_rate:>7.2%}  (expect ~94.00%)")
print(f"{'Cetacean':<20} | {n_cetacean:>5} | {cetacean_rate:>7.2%}  (expect 100.00%)")

Dataset              |     n |     Rate
Negative controls    |   437 |  46.22%  (expect ~46.22%)
Prawn                |    82 | 100.00%  (expect 100.00%)
Tuna                 |   200 |  94.50%  (expect ~94.00%)
Cetacean             |   200 |  99.50%  (expect 100.00%)


In [13]:
# --- Cell 7: Save the final config and bucket reference as SEPARATE files for Stage 2 ---

stage1_final_config = {
    "img_size": list(IMG_SIZE),
    "n_segments": N_SEGMENTS,
    "compactness": COMPACTNESS,
    "glcm_levels": GLCM_LEVELS,
    "glint_thresh": GLINT_THRESH,
    "n_buckets": N_BUCKETS,
    "bucket_edges": bucket_edges.tolist(),
    "z_thresh": Z_THRESH,
    "min_features_flagged": MIN_FEATURES_FLAGGED,
    "min_component_area_px": MIN_COMPONENT_AREA_PX,
    "local_z_thresh": LOCAL_Z_THRESH,
    "min_area_local": MIN_AREA_LOCAL,
    "false_positive_rate": false_positive_rate,
    "anomaly_detection_rate_on_prawn_images": prawn_rate,
    "anomaly_detection_rate_on_tuna_images": tuna_rate,
    "anomaly_detection_rate_on_cetacean_images": cetacean_rate,
    "n_negative_images": n_negative,
    "n_prawn_images": n_prawn,
    "n_tuna_images_sampled": n_tuna,
    "n_cetacean_images_sampled": n_cetacean,
    "note": ("Detection is species-agnostic: combines a texture rule (5-feature GLCM/LBP, "
             "bucket-relative z-scores) OR a local-brightness rule (within-image relative, "
             "no external baseline). Rates measure whether the detector flags a region as "
             "anomalous, NOT species-level identification - that is Stage 3's job. Stage 2 "
             "must apply BOTH rules (texture OR local-brightness) to reproduce this detector "
             "exactly - using only the texture rule will not match these numbers."),
}

config_path = OUTPUT_DIR / "stage1_final_config.json"
with open(config_path, "w") as f:
    json.dump(stage1_final_config, f, indent=2)

bucket_ref_path = OUTPUT_DIR / "stage1_bucket_ref.pkl"
with open(bucket_ref_path, "wb") as f:
    pickle.dump(bucket_ref, f)

print(f"Saved: {config_path}")
print(f"Saved: {bucket_ref_path}")
print(f"\n{json.dumps(stage1_final_config, indent=2)}")

Saved: stage1_output\stage1_final_config.json
Saved: stage1_output\stage1_bucket_ref.pkl

{
  "img_size": [
    512,
    512
  ],
  "n_segments": 250,
  "compactness": 20,
  "glcm_levels": 32,
  "glint_thresh": 220,
  "n_buckets": 3,
  "bucket_edges": [
    98.48828532025743,
    132.1124305725098
  ],
  "z_thresh": 2.5,
  "min_features_flagged": 2,
  "min_component_area_px": 2000,
  "local_z_thresh": 2.5,
  "min_area_local": 1000,
  "false_positive_rate": 0.4622425629290618,
  "anomaly_detection_rate_on_prawn_images": 1.0,
  "anomaly_detection_rate_on_tuna_images": 0.945,
  "anomaly_detection_rate_on_cetacean_images": 0.995,
  "n_negative_images": 437,
  "n_prawn_images": 82,
  "n_tuna_images_sampled": 200,
  "n_cetacean_images_sampled": 200,
  "note": "Detection is species-agnostic: combines a texture rule (5-feature GLCM/LBP, bucket-relative z-scores) OR a local-brightness rule (within-image relative, no external baseline). Rates measure whether the detector flags a region as anomal